In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, accuracy_score
from imblearn.under_sampling import RandomUnderSampler

import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, Input, Dropout
from keras.models import Sequential
import keras_tuner

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, mean_absolute_error, mean_squared_error

print("1. Cargando y reestructurando el dataset a CUATRO columnas...")
df = pd.read_csv("dataset_elpino.csv", sep=";") # Ajustar sep si es necesario

df['GRD_Target'] = df['GRD'].astype(str).str.split(' - ').str[0].str.strip()

def limpiar_codigo(valor):
    valor_str = str(valor).strip()
    if valor_str and valor_str != '-' and valor_str != 'nan':
        return valor_str.split(' - ')[0].strip().replace(" ", "_")
    return ""

def unir_secundarios(fila):
    codigos = [limpiar_codigo(v) for v in fila if limpiar_codigo(v) != ""]
    return " ".join(codigos)

# Jerarquía
df['Diag_Principal'] = df['Diag 01 Principal (cod+des)'].apply(limpiar_codigo)
df['Proc_Principal'] = df['Proced 01 Principal (cod+des)'].apply(limpiar_codigo)
cols_diag_sec = [col for col in df.columns if 'Diag' in col and 'Secundario' in col]
cols_proc_sec = [col for col in df.columns if 'Proced' in col and 'Secundario' in col]
df['Diag_Secundarios'] = df[cols_diag_sec].apply(unir_secundarios, axis=1)
df['Proc_Secundarios'] = df[cols_proc_sec].apply(unir_secundarios, axis=1)

# Poda Clínica
MIN_PACIENTES = 5
conteos_grd = df['GRD_Target'].value_counts()
clases_validas = conteos_grd[conteos_grd >= MIN_PACIENTES].index
df_filtrado = df[df['GRD_Target'].isin(clases_validas)].copy()

X = df_filtrado[['Edad en años', 'Sexo (Desc)', 'Diag_Principal', 'Proc_Principal', 'Diag_Secundarios', 'Proc_Secundarios']]
y = df_filtrado['GRD_Target']

# ==========================================
# PREPARACIÓN PARA KERAS
# ==========================================
# Keras necesita que el target sea un número entero (0 a N_clases)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
print(f"Total de GRD a predecir: {num_classes}")

preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Edad en años']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Sexo (Desc)']),
        ('diag_prin', CountVectorizer(token_pattern=r"(?u)\b[\w\.]+\b"), 'Diag_Principal'),
        ('proc_prin', CountVectorizer(token_pattern=r"(?u)\b[\w\.]+\b"), 'Proc_Principal'),
        ('diag_sec', TfidfVectorizer(token_pattern=r"(?u)\b[\w\.]+\b"), 'Diag_Secundarios'),
        ('proc_sec', TfidfVectorizer(token_pattern=r"(?u)\b[\w\.]+\b"), 'Proc_Secundarios')
    ]
)

print("2. Preprocesando y balanceando los datos...")
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Transformamos matemáticamente los textos
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep  = preprocesador.transform(X_test)

# Submuestreo
MAX_PACIENTES = 150
conteos_train = pd.Series(y_train).value_counts()
estrategia_recorte = {clase: min(conteo, MAX_PACIENTES) for clase, conteo in conteos_train.items()}
recortador = RandomUnderSampler(sampling_strategy=estrategia_recorte, random_state=42)

X_train_res, y_train_res = recortador.fit_resample(X_train_prep, y_train)

# Convertir a matriz densa para Keras
X_train_res_dense = X_train_res.toarray()
X_test_dense = X_test_prep.toarray()
input_shape = X_train_res_dense.shape[1]

# ==========================================
# FASE KERAS TUNER: BÚSQUEDA DE ARQUITECTURA
# ==========================================
print("\n3. Configurando Keras Tuner...")

def build_model(hp):
    modelo = Sequential()
    modelo.add(Input(shape=(input_shape,)))
    
    # El Tuner probará tener entre 1 y 3 capas
    for i in range(hp.Choice("capas", [1, 2, 3])):
        # El Tuner probará diferentes cantidades de neuronas
        modelo.add(Dense(
            units=hp.Choice(f"neuronas_capa_{i}", [64, 128, 256, 512]), 
            activation="relu"
        ))
        # Agregamos Dropout opcional para evitar memorización
        modelo.add(Dropout(hp.Choice(f"dropout_capa_{i}", [0.2, 0.3, 0.5])))
        
    # Capa de salida Multiclase
    modelo.add(Dense(num_classes, activation="softmax"))
    
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])),
        loss="sparse_categorical_crossentropy",
        metrics=['accuracy']
    )
    return modelo

tuner = keras_tuner.RandomSearch(
    hypermodel=build_model,
    objective=keras_tuner.Objective("val_accuracy", direction="max"),
    max_trials=10, # Hará 10 pruebas diferentes de redes. Puedes subirlo si tienes tiempo.
    executions_per_trial=1,
    overwrite=True,
    directory="modelos_hospital",
    project_name="grd_optimizacion"
)

print("\n4. ¡Iniciando el entrenamiento y búsqueda del mejor modelo!...")
# Parada temprana: si la red deja de aprender tras 3 intentos, aborta y sigue con la próxima
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

tuner.search(
    X_train_res_dense, y_train_res, 
    epochs=15, 
    validation_data=(X_test_dense, y_test),
    callbacks=[stop_early],
    batch_size=64
)

# Extraer al campeón
mejor_modelo = tuner.get_best_models()[0]

print("\n--- RESUMEN DE LA MEJOR ARQUITECTURA ENCONTRADA ---")
mejor_modelo.summary()
mejor_modelo.save("mejor_modelo_grd.keras")

# ==========================================
# EVALUACIÓN FINAL
# ==========================================
print("\n5. Evaluando el mejor modelo...")
# Predicciones de la red neuronal
y_pred_probs = mejor_modelo.predict(X_test_dense)
y_pred = np.argmax(y_pred_probs, axis=1) # Elige la clase con mayor probabilidad

# 1. Métricas de Clasificación (Usamos 'macro' por el desbalance extremo)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1_mac = f1_score(y_test, y_pred, average='macro', zero_division=0)

# 2. Métricas de Regresión solicitadas (MAE y MSE)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("\n" + "=" * 45)
print("     RESULTADOS FINALES DEL MODELO (Keras)     ")
print("=" * 45)
print("Métricas de Clasificación:")
print(f" Accuracy General: {acc:.4f}  ({acc * 100:.2f}%)")
print(f" Precision Macro:  {prec:.4f}  ({prec * 100:.2f}%)")
print(f" Recall Macro:     {rec:.4f}  ({rec * 100:.2f}%)")
print(f" F1-Score Macro:   {f1_mac:.4f}  ({f1_mac * 100:.2f}%)")
print("-" * 45)
print("Métricas de Distancia/Regresión:")
print(f" MAE (Error Absoluto Medio):   {mae:.4f}")
print(f" MSE (Error Cuadrático Medio): {mse:.4f}")
print("=" * 45)

Trial 10 Complete [00h 01m 12s]
val_accuracy: 0.8058012127876282

Best val_accuracy So Far: 0.8058012127876282
Total elapsed time: 00h 07m 13s

--- RESUMEN DE LA MEJOR ARQUITECTURA ENCONTRADA ---


/home/alex/Documentos/UNAB/Aprendizaje de maquina/Trabajos/Fase 1/.venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │     2,723,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 327)            │        42,183 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,831,175 (10.80 MB)

 Trainable params: 2,831,175 (10.80 MB)

 Non-trainable params: 0 (0.00 B)


5. Evaluando el mejor modelo...
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

     RESULTADOS FINALES DEL MODELO (Keras)     
Métricas de Clasificación:
 Accuracy General: 0.8058  (80.58%)
 Precision Macro:  0.6151  (61.51%)
 Recall Macro:     0.5905  (59.05%)
 F1-Score Macro:   0.5833  (58.33%)
---------------------------------------------
Métricas de Distancia/Regresión:
 MAE (Error Absoluto Medio):   5.0481
 MSE (Error Cuadrático Medio): 669.8560
